In [51]:
from itertools import accumulate, takewhile
from random import uniform
import scipy as sp
import numpy as np
from plotly import graph_objects as go

In [52]:
class discrete_distribution:
    def __init__(self, domain: list[int], probabilities: list[float]) -> None:
        assert len(domain) == len(probabilities), "Probability must be defined for each domain element"
        prob_sum = sum(probabilities)
        self.pmf = [probability/prob_sum for probability in probabilities]
        self.cdf = [cdf for cdf in accumulate(self.pmf)]
        self.domain = domain
    
    def sample(self) -> int:
        variate = uniform(a=0.0, b=1.0)
        
        for item, cdf in zip(self.domain, self.cdf):
            if cdf >= variate:
                return item
            
        raise ValueError("Cannot sample the distribution!")


In [ ]:
class NormalInverseGamma:
    def __init__(self, mean: float, variance: float, n_degrees_of_freedom: float, n_pseudoobservations: float) -> None:
        self.distribution = sp.stats.normal_inverse_gamma(a=n_degrees_of_freedom/2, 
                                                          b=n_degrees_of_freedom*variance/2, 
                                                          mu=mean, 
                                                          lmbda=n_pseudoobservations)
        
    def sample_likelihood(self, sample: float) -> float:
        """Calculate sample marginal likelihood.

        Args:
            sample: random sample

        Returns:
            sample marginal likelihood given the distribution parameters (the variance parameter is 
            marginalized)
        """

        return sp.stats.t()

In [73]:
# data simulation
# for now I choose some simple model with two states (two distinct sets of distribution parameters) 
# where the switching process will be modelled as a discrete markov chain
n_state = 2
transition_dist: list[discrete_distribution] = [discrete_distribution([0, 1], [0.99, 0.01]), 
                   discrete_distribution([0, 1], [0.01, 0.99])]
init_dist = discrete_distribution([0, 1], [0.5, 0.5])

# arbitrary values of the hyperparameters
mu_prior = 0.0
n_prior = 1
nu_prior = 10
sigma_prior = 2

variance_prior = sp.stats.invgamma(a=nu_prior/2, scale=nu_prior*sigma_prior**2/2)
state_distributions = []

for i_state in range(n_state):
    variance = variance_prior.rvs()
    print(variance)
    mean_prior = sp.stats.norm(loc=mu_prior, scale=np.sqrt(variance)/n_prior)
    mean = mean_prior.rvs()
    print(mean)
    state_distributions.append(sp.stats.norm(loc=mean, scale=np.sqrt(variance)))

n_step = 1000
chain = []
chain.append(init_dist.sample())
samples = []
samples.append(state_distributions[chain[0]].rvs())

for i_step in range(1, n_step):
    chain.append(transition_dist[chain[i_step-1]].sample())
    samples.append(state_distributions[chain[i_step]].rvs())

fig_obj = go.Figure()
fig_obj.add_trace(go.Scatter(y=chain, mode="markers+lines", name="True state"))
fig_obj.show()

fig_obj = go.Figure()
fig_obj.add_trace(go.Scatter(y=samples, mode="markers+lines", name="Samples"))
fig_obj.show()


3.481114127947811
1.1627567914952717
1.9003856704870183
-2.120479399413867


In [74]:
# Initialization
run_length_prior = discrete_distribution([0], [1.0])
